# Training_Best_AdamW_vs_Prodigy_30ep.ipynb

**Branch**: `Visualizer_Building` (post-merge consolidato).

**Obiettivo**: training rigoroso a **30 epoche** delle 4 architetture **realmente migliori** dai `results/`, con sweep optimizer **mirato per-arch** (la config Prodigy generica fallisce). Naming chiaro, simulator integrato, analisi su **val_total** (loss combinata PINN), non solo val_data.

## Selezione architetture -- verità dai results/

Ranking effettivo dai 15-epoch runs (AdamW lr=2e-3 + OneCycle, h=32 r=8, Po2=ON, PINN-on, OU=0, highway):

| Tag | Variant | Params | val_data ep15 |
|---|---|---:|---:|
| **A8_ATTN** | `attn` | 3,936 | **0.1795** ← BEST architettonico |
| **A3_STACKED_SKIP** | `stacked_2_skip` | 2,624 | 0.2164 |
| **A1_BASELINE** | `baseline` | 864 | 0.2205 (production reference) |
| **EVPROP_ALIF** | `eventprop_alif_full` | 864 | 0.2226 (best 5ep, AdamW 2e-3) |

> F7 (no-Po2) → **escluso**: Po2 è vincolo FPGA non-negoziabile. La causa del floor era OU, non Po2 (confermato P14). OU=0 (segnali puliti) è accettato come setup.

## Sweep optimizer per-arch (NON config generica)

Dal sweep 4×11 (`EVENTPROP_OPTIMIZER_SWEEP.md`) emerge che **Prodigy lr=1.0 d=1.0 è frozen/divergente** in 10/11 config. Le sole config Prodigy che funzionano sono **specifiche per metodo**:

| Architettura | AdamW (config storica vincente) | Prodigy (UNICA decente per quell'arch) |
|---|---|---|
| A1_BASELINE / A8_ATTN / A3_STACKED_SKIP | lr=2e-3 + OneCycleLR | **lr=0.1, d_coef=1.0** + sched=none |
| EVPROP_ALIF (adjoint fragile) | lr=2e-3 + OneCycleLR | **lr=1.0, d_coef=0.1** (brake forte) + sched=none |

Razionale Prodigy diversa per EventProp: nel sweep, EVPROP_ALIF con `prodigy_lr01_d10` divergeva (val_data 0.97). Solo `prodigy_lr1_d01` ha dato 0.2353 (terzo best). Per BPTT methods invece `prodigy_lr01_d10` ha dato 0.2226 (secondo best per baseline).

**Tag completo**: `T30_<ARCH>_<OPT>` → es. `T30_A8_ATTN_adamw`, `T30_EVPROP_ALIF_prodigy`. Totale **4 arch × 2 opt = 8 run**.

## Configurazione comune (matches P15_S2E)

- 30 epoche × 190 step (= 5700 batch)
- batch=8, val_batch=64, seq_len=50
- scenario_mix=highway, cut_in=0.0, **noise_scale=0** (segnali puliti, accettato)
- cache: `data/cache_1500_highway_cut0.0_ou0.0.pt`
- h=32, r=8, **po2_enabled=1 sempre** (FPGA constraint)
- Lambdas PINN (matches P15): **data=1.0, phys=0.1, ou=0.05, bc=1.0, sr=0**

## Fix Prodigy lr_eff (già fatto nel commit precedente)

`train.py` logga per-epoca `prodigy_d`, `prodigy_d_max`, `prodigy_lr_eff = lr × d`. `plot_g3_lr_schedule` auto-detect Prodigy → 2-panel.

## Output e analisi

- `results/T30_<TAG>/` per ogni run (CSV epoca + batch + plot G1-G13 + best_model.pt)
- **Cella 5 analisi**: pivot **val_total** (loss combinata PINN) 4×2 — PRIMARY metric. `val_data` come secondary (sub-metric).
- Cella 6 trajectory overlay 8 curve (val_total + val_data)
- Cella 7 simulator: CFSimulator su mixed cut-in cache per ogni best checkpoint

**Tempi Azure stimati**: ~6h totali (8 runs × ~45 min avg). Adatto a esecuzione overnight.

**SKIP_IF_EXISTS**: rilancio sicuro dopo interruzioni. Push per-run.

In [ ]:
# ===========================================================
# CELLA 0 -- Bootstrap deps + git checkout Visualizer_Building
# ===========================================================
import sys

print('Installing/upgrading dependencies (idempotent)...')
!{sys.executable} -m pip install --quiet matplotlib pandas pillow
!{sys.executable} -m pip install --quiet nbstripout
!{sys.executable} -m pip install --quiet prodigyopt   # per --optimizer prodigy
!nbstripout --install --attributes .gitattributes 2>/dev/null
print('   [OK] deps installed + nbstripout attivato')

print('\ngit fetch + checkout Visualizer_Building + pull:')
!git fetch origin
!git checkout Visualizer_Building 2>/dev/null || git checkout -b Visualizer_Building origin/Visualizer_Building
!git pull --no-rebase --no-edit origin Visualizer_Building

print('\nBranch + commit attuale:')
!git branch --show-current
!git log --oneline -3

In [ ]:
# ===========================================================
# CELLA 1 -- ENV CHECK + verifica train.py CLI + build_model 4 varianti
# ===========================================================
import sys, os, platform, subprocess

print(f'Python:          {sys.version.split()[0]} ({platform.system()})')
print(f'Working dir:     {os.getcwd()}')

checks = []
for mod in ['torch', 'numpy', 'pandas', 'matplotlib', 'prodigyopt']:
    try:
        m = __import__(mod)
        v = getattr(m, '__version__', '?')
        print(f'   [OK] {mod:<14} v{v}')
        checks.append((mod, True))
    except Exception as e:
        print(f'   [FAIL] {mod:<14} {e}')
        checks.append((mod, False))

import torch
print(f'\nCUDA: ' + ('[OK] ' + torch.cuda.get_device_name(0) if torch.cuda.is_available() else '[INFO] CPU only'))

print('\nFile critici:')
for f in ['train.py', 'config.py', 'core/network.py', 'core/eventprop.py',
          'utils/plot_diagnostics.py', 'utils/simulator/engine.py']:
    exists = os.path.isfile(f)
    print(f'   {"[OK]" if exists else "[MISSING]"} {f}')
    checks.append((f, exists))

# Verifica CLI flags critici
res = subprocess.run([sys.executable, 'train.py', '--help'], capture_output=True, text=True)
help_text = res.stdout + res.stderr
for flag in ['--training_method', '--prodigy_d_coef', '--noise_scale', '--po2_enabled',
             '--max_steps_per_epoch', '--cf_hidden_size', '--cf_rank']:
    ok = flag in help_text
    print(f'   {"[OK]" if ok else "[MISSING]"} train.py supporta {flag}')
    checks.append((flag, ok))

# Verifica le 4 varianti reali che useremo (matches results/P15_S2E_*)
from core.network import build_model
for v in ['baseline', 'attn', 'stacked_2_skip', 'eventprop_alif_full']:
    try:
        m = build_model(v)
        n = sum(p.numel() for p in m.parameters())
        print(f'   [OK] build_model({v:25s}) params={n:>5,d}')
    except Exception as e:
        print(f'   [FAIL] build_model({v}): {e}')
        checks.append((v, False))

n_fail = sum(1 for _, ok in checks if not ok)
print(f'\n{"[OK]" if n_fail == 0 else f"[FAIL] {n_fail} problemi"} ENV CHECK')
if n_fail > 0:
    raise SystemExit('Env check failed')

In [ ]:
# ===========================================================
# CELLA 2 -- TRAIN_PLAN (4 arch reali x 2 opt = 8 run, sweep per-arch)
# ===========================================================
# Architetture: le 4 realmente migliori dai results/ (P15_S2E_*).
# Optimizer: AdamW lr=2e-3 + OneCycle storico vincente PER TUTTE,
# Prodigy lr/d_coef SPECIFICA per arch (la generica fallisce, vedi
# EVENTPROP_OPTIMIZER_SWEEP.md):
#   - BPTT methods (A1/A8/A3): Prodigy lr=0.1 d=1.0 -> unica decente
#   - EventProp adjoint:        Prodigy lr=1.0 d=0.1 -> unica decente (brake)
# Tutte: Po2=ON (FPGA constraint), PINN-on (matches P15 lambdas), OU=0.

import time, subprocess, sys, os, datetime, shutil, glob, json
import pandas as pd, numpy as np

COMMON = {
    'epochs':              30,
    'max_steps_per_epoch': 190,
    'batch_size':          8,
    'val_batch_size':      64,
    'seq_len':             50,
    'scenario_mix':        'highway',
    'cut_in_ratio':        0.0,
    'cf_hidden_size':      32,
    'cf_rank':             8,
    'noise_scale':         0.0,        # segnali puliti (OU off, accettato)
    'n_train':             1500,
    'n_val':               300,
    'max_inf_streak':      99999,      # no abort
    'early_stop_patience': 0,          # no early stop, run full 30 ep
    # PINN ATTIVO (matches P15_S2E -- NON solo data):
    'po2_enabled':         1,          # FPGA constraint, sempre ON
    'lambda_data':         1.0,
    'lambda_phys':         0.1,
    'lambda_ou':           0.05,
    'lambda_bc':           1.0,
    'lambda_sr':           0.0,
}
CACHE_PATH = f'data/cache_{COMMON["n_train"]}_{COMMON["scenario_mix"]}_cut{COMMON["cut_in_ratio"]}_ou{COMMON["noise_scale"]}.pt'

# 4 ARCHITETTURE realmente top (matches results/P15_S2E_*):
ARCHITECTURES = {
    'A1_BASELINE': {
        'training_method': 'baseline',
        '_params':         864,
        '_p15_val_data':   0.2205,
        '_desc':           'A1 baseline ALIF (production reference, 864p)',
    },
    'A8_ATTN': {
        'training_method': 'attn',
        '_params':         3936,
        '_p15_val_data':   0.1795,
        '_desc':           'A8 Spike Attention (3936p) -- BEST architettonico',
    },
    'A3_STACKED_SKIP': {
        'training_method': 'stacked_2_skip',
        '_params':         2624,
        '_p15_val_data':   0.2164,
        '_desc':           'A3 Stacked-2 + skip connection (2624p)',
    },
    'EVPROP_ALIF': {
        'training_method': 'eventprop_alif_full',
        '_params':         864,
        '_p15_val_data':   0.2226,
        '_desc':           'A1 + EventProp adjoint (training method nuovo, 864p)',
    },
}

# OPTIMIZER per-arch (NON config generica). AdamW e' la stessa per tutte,
# Prodigy cambia in funzione del metodo (BPTT vs EventProp).
def _opt_for(arch_name, opt_name):
    """Restituisce dict config optimizer specifico per (arch, opt).

    Razionale: nel sweep 4x11, Prodigy lr=1.0 d=1.0 e' frozen/divergente
    in 10/11 config. Le sole config che funzionano sono:
    - per BPTT (baseline/attn/stacked): Prodigy lr=0.1 d=1.0 -> 0.2226 baseline
    - per EventProp adjoint:            Prodigy lr=1.0 d=0.1 -> 0.2353 (brake)
    """
    if opt_name == 'adamw':
        return {
            'optimizer':      'adamw',
            'lr':             2e-3,
            'scheduler':      'onecycle',
            'max_lr':         2e-3,
            'prodigy_d_coef': 1.0,        # ignored
            '_opt_desc':      'AdamW lr=2e-3 + OneCycleLR (config storica vincente)',
        }
    if opt_name == 'prodigy':
        if arch_name == 'EVPROP_ALIF':
            return {
                'optimizer':      'prodigy',
                'lr':             1.0,        # canonical Prodigy lr
                'scheduler':      'none',
                'max_lr':         1.0,        # unused
                'prodigy_d_coef': 0.1,        # BRAKE FORTE (unica decente per EventProp)
                '_opt_desc':      'Prodigy lr=1.0 d_coef=0.1 (brake, unica decente per EventProp)',
            }
        # BPTT methods (baseline/attn/stacked):
        return {
            'optimizer':      'prodigy',
            'lr':             0.1,            # LR ridotta (unica decente per BPTT)
            'scheduler':      'none',
            'max_lr':         0.1,            # unused
            'prodigy_d_coef': 1.0,            # canonical
            '_opt_desc':      'Prodigy lr=0.1 d_coef=1.0 (unica decente per BPTT)',
        }
    raise ValueError(f'opt sconosciuta: {opt_name}')

# Build full plan: 4 arch x 2 opt = 8 runs
TRAIN_PLAN = []
for arch_name, arch_cfg in ARCHITECTURES.items():
    for opt_name in ['adamw', 'prodigy']:
        opt_cfg = _opt_for(arch_name, opt_name)
        tag = f'T30_{arch_name}_{opt_name}'
        full_cfg = {**COMMON, **arch_cfg, **opt_cfg, 'tag': tag}
        TRAIN_PLAN.append(full_cfg)

# Filter run list (modifica per disabilitare singoli plan)
RUN_TAGS = set(p['tag'] for p in TRAIN_PLAN)  # default: TUTTI

print(f'TRAIN PLAN: {len(TRAIN_PLAN)} run totali (4 arch x 2 opt)')
print(f'Cache: {CACHE_PATH}  ({"esistente" if os.path.isfile(CACHE_PATH) else "DA GENERARE (Cella 3)"})')
print(f'Setup comune: 30ep x 190step, batch=8, h=32 r=8, Po2=ON, PINN-on (data=1, phys=0.1, ou=0.05, bc=1.0, sr=0)')
print(f'Scenario: highway, cut_in=0.0, noise_scale=0 (segnali puliti)')
print()
for p in TRAIN_PLAN:
    in_run = '[RUN]' if p['tag'] in RUN_TAGS else '[SKIP]'
    print(f'{in_run} {p["tag"]:<32s}  params={p["_params"]:>5,d}  p15_val_data_ref={p["_p15_val_data"]:.4f}')
    print(f'         arch  : {p["_desc"]}')
    print(f'         opt   : {p["_opt_desc"]}')
    print(f'         CLI   : --training_method {p["training_method"]} --optimizer {p["optimizer"]} '
          f'--lr {p["lr"]} --scheduler {p["scheduler"]} --prodigy_d_coef {p["prodigy_d_coef"]}')
    print()

In [ ]:
# ===========================================================
# CELLA 3 -- Cache check + auto-generate if missing
# ===========================================================
if not os.path.isfile(CACHE_PATH):
    print(f'[CACHE GEN] {CACHE_PATH} mancante, generazione automatica...')
    from data.generator import generate_dataset
    from config import SEED
    import torch
    t0 = time.time()
    train_data = generate_dataset(COMMON['n_train'], base_seed=SEED,
                                    scenario_mix={'highway': 1.0},
                                    cut_in_ratio=COMMON['cut_in_ratio'],
                                    noise_scale=COMMON['noise_scale'])
    val_data = generate_dataset(COMMON['n_val'], base_seed=SEED + 1,
                                  scenario_mix={'highway': 1.0},
                                  cut_in_ratio=COMMON['cut_in_ratio'],
                                  noise_scale=COMMON['noise_scale'])
    torch.save({'train': train_data, 'val': val_data, 'seed': SEED}, CACHE_PATH)
    print(f'[CACHE GEN] saved {CACHE_PATH} ({os.path.getsize(CACHE_PATH)/1024/1024:.1f} MB) in {time.time()-t0:.0f}s')
else:
    sz = os.path.getsize(CACHE_PATH) / 1024 / 1024
    print(f'[CACHE OK] {CACHE_PATH} esistente ({sz:.1f} MB)')

In [ ]:
# ===========================================================
# CELLA 4 -- RUN sweep loop (SKIP_IF_EXISTS + push per-run)
# ===========================================================
# SKIP_IF_EXISTS=True: salta plan con results/<tag>/training_log.csv esistente
# Permette di rilanciare la cella dopo crash/interruzioni senza ripetere lavoro.
SKIP_IF_EXISTS = True

def _build_cli(p):
    args = [
        sys.executable, 'train.py',
        '--training_method',     p['training_method'],
        '--epochs',              str(p['epochs']),
        '--n_train',             str(p['n_train']),
        '--n_val',               str(p['n_val']),
        '--batch_size',          str(p['batch_size']),
        '--val_batch_size',      str(p['val_batch_size']),
        '--seq_len',             str(p['seq_len']),
        '--scheduler',           p['scheduler'],
        '--max_lr',              str(p['max_lr']),
        '--lr',                  str(p['lr']),
        '--optimizer',           p['optimizer'],
        '--prodigy_d_coef',      str(p['prodigy_d_coef']),
        '--scenario_mix',        p['scenario_mix'],
        '--cut_in_ratio',        str(p['cut_in_ratio']),
        '--cf_hidden_size',      str(p['cf_hidden_size']),
        '--cf_rank',             str(p['cf_rank']),
        '--lambda_data',         str(p['lambda_data']),
        '--lambda_phys',         str(p['lambda_phys']),
        '--lambda_ou',           str(p['lambda_ou']),
        '--lambda_bc',           str(p['lambda_bc']),
        '--lambda_sr',           str(p['lambda_sr']),
        '--noise_scale',         str(p['noise_scale']),
        '--po2_enabled',         str(p['po2_enabled']),
        '--max_inf_streak',      str(p['max_inf_streak']),
        '--early_stop_patience', str(p['early_stop_patience']),
        '--max_steps_per_epoch', str(p['max_steps_per_epoch']),
        '--data_cache',          CACHE_PATH,
        '--tag',                 p['tag'],
    ]
    return args

def _push_run(p):
    tag = p['tag']
    src, dst = f'checkpoints/{tag}', f'results/{tag}'
    if not os.path.isdir(src):
        print(f'   [WARN] {src} mancante, niente da pushare')
        return False
    if os.path.isdir(dst): shutil.rmtree(dst)
    os.makedirs(f'{dst}/plots', exist_ok=True)
    for f in glob.glob(f'{src}/*.csv') + glob.glob(f'{src}/*.json'):
        shutil.copy2(f, dst)
    for f in glob.glob(f'{src}/plots/*.png'):
        shutil.copy2(f, f'{dst}/plots/')

    val_str = ''
    if os.path.isfile(f'{dst}/training_log.csv'):
        edf = pd.read_csv(f'{dst}/training_log.csv')
        if len(edf) > 0:
            best_idx = int(edf.val_data.idxmin())
            val_str = f'best val_data={edf.val_data.min():.4f} (E{best_idx+1}/{len(edf)})'

    ts = datetime.datetime.now().strftime('%Y-%m-%d %H:%M')
    msg = (
        f'results (T30 Best AdamW vs Prodigy 30ep): {tag} ({ts})\n\n'
        f'{val_str}\n'
        f'method={p["training_method"]} optimizer={p["optimizer"]} '
        f'lr={p["lr"]} sched={p["scheduler"]} po2={p["po2_enabled"]}\n'
        f'Branch Visualizer_Building\n'
    )
    with open('/tmp/t30_msg.txt', 'w', encoding='utf-8') as f:
        f.write(msg)
    !git add {dst}
    !git commit -F /tmp/t30_msg.txt
    !rm /tmp/t30_msg.txt
    !git pull --no-rebase --no-edit origin Visualizer_Building
    !git push origin Visualizer_Building
    return True

# ── Esecuzione ────────────────────────────────────
run_results = []
t_start = time.time()
run_idx = 0
total_runs = sum(1 for p in TRAIN_PLAN if p['tag'] in RUN_TAGS)

for p in TRAIN_PLAN:
    if p['tag'] not in RUN_TAGS:
        continue
    run_idx += 1
    tag = p['tag']

    if SKIP_IF_EXISTS and os.path.isfile(f'results/{tag}/training_log.csv'):
        try:
            ep = pd.read_csv(f'results/{tag}/training_log.csv')
            v_str = f'val_data={ep.val_data.min():.4f}' if len(ep) else 'empty'
        except Exception:
            v_str = 'unreadable'
        print(f'\n[{run_idx}/{total_runs}] [SKIP_EXIST] {tag}: ({v_str}). '
              f'Set SKIP_IF_EXISTS=False per ri-eseguire.')
        run_results.append({'tag': tag, 'status': 'skipped', 'elapsed': 0.0})
        continue

    print('\n' + '=' * 78)
    print(f'[{run_idx}/{total_runs}] {tag}')
    print(f'  {p["_desc"]}')
    print(f'  optimizer={p["optimizer"]} lr={p["lr"]} sched={p["scheduler"]} po2={p["po2_enabled"]}')
    print('=' * 78)
    cmd = _build_cli(p)
    t0 = time.time()
    res = subprocess.run(cmd, capture_output=False)
    elapsed = time.time() - t0
    status = 'ok' if res.returncode == 0 else f'fail({res.returncode})'
    elapsed_total = time.time() - t_start
    eta_min = (elapsed_total / run_idx) * (total_runs - run_idx) / 60
    print(f'\n[{run_idx}/{total_runs}] {tag} -> {status} in {elapsed/60:.1f} min   '
          f'total={elapsed_total/60:.0f}min  ETA={eta_min:.0f}min')
    print(f'\nPush results/{tag}...')
    _push_run(p)
    run_results.append({'tag': tag, 'status': status, 'elapsed': elapsed})

print(f'\n{"="*78}\nSWEEP TRAINING COMPLETO in {(time.time()-t_start)/60:.0f} min\n{"="*78}')
for r in run_results:
    if r['status'] != 'skipped':
        print(f"   {r['tag']:<35} {r['status']:<15} {r['elapsed']/60:.1f}min")

In [ ]:
# ===========================================================
# CELLA 5 -- ANALISI: pivot val_total (PINN loss combinata) 4x2 + sub-metric
# ===========================================================
# PRIMARY metric = val_total (somma data + phys + ou + bc + sr) -- la vera
# loss minimizzata dalla rete. val_data e' UNA delle sub-metric (RMSE accel
# masked) e da sola non rappresenta il comportamento PINN multi-objective.
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ARCH_ORDER = ['A1_BASELINE', 'A8_ATTN', 'A3_STACKED_SKIP', 'EVPROP_ALIF']
OPT_ORDER  = ['adamw', 'prodigy']

rows = []
for p in TRAIN_PLAN:
    tag = p['tag']
    arch = tag.replace('T30_', '').rsplit('_', 1)[0]
    opt = tag.rsplit('_', 1)[1]
    log = f'results/{tag}/training_log.csv'
    if not os.path.isfile(log):
        rows.append({'tag': tag, 'arch': arch, 'opt': opt, 'n_ep': 0})
        continue
    ep = pd.read_csv(log)
    if len(ep) == 0:
        rows.append({'tag': tag, 'arch': arch, 'opt': opt, 'n_ep': 0})
        continue
    # Best per val_total (PRIMARY) e per val_data (secondary)
    best_total_idx = int(ep.val_total.idxmin())
    best_data_idx  = int(ep.val_data.idxmin())
    row = {
        'tag': tag, 'arch': arch, 'opt': opt, 'n_ep': len(ep),
        # PRIMARY
        'val_total_best':    float(ep.val_total.min()),
        'best_ep_total':     best_total_idx + 1,
        'val_total_last':    float(ep.val_total.iloc[-1]),
        # Sub-metric AL best_ep_total (coerenti tra loro, stessa epoca)
        'val_data_at_best':  float(ep.val_data.iloc[best_total_idx]),
        'val_phys_at_best':  float(ep.val_phys.iloc[best_total_idx]) if 'val_phys' in ep.columns else float('nan'),
        'val_ou_at_best':    float(ep.val_ou.iloc[best_total_idx])   if 'val_ou' in ep.columns else float('nan'),
        'val_bc_at_best':    float(ep.val_bc.iloc[best_total_idx])   if 'val_bc' in ep.columns else float('nan'),
        # Secondary: best val_data assoluto (puo' essere a un'epoca diversa)
        'val_data_best':     float(ep.val_data.min()),
        'best_ep_data':      best_data_idx + 1,
        # Activity
        'spike_rate_last':   float(ep.spike_rate.iloc[-1]),
        # Prodigy adapter snapshot
        'prodigy_lr_eff_last': float(ep['prodigy_lr_eff'].iloc[-1]) if 'prodigy_lr_eff' in ep.columns and not pd.isna(ep['prodigy_lr_eff'].iloc[-1]) else None,
    }
    rows.append(row)

df_runs = pd.DataFrame(rows)
df_done = df_runs[df_runs.n_ep > 0]

if len(df_done) == 0:
    print('Nessun risultato disponibile -- esegui Cella 4 prima.')
else:
    # PRIMARY: val_total best
    pivot_total = df_done.pivot(index='arch', columns='opt', values='val_total_best').reindex(ARCH_ORDER)[OPT_ORDER]
    display(Markdown('## 1. val_total BEST (PRIMARY -- loss PINN combinata: data + phys + ou + bc)'))
    display(pivot_total.round(4))

    pivot_total_ep = df_done.pivot(index='arch', columns='opt', values='best_ep_total').reindex(ARCH_ORDER)[OPT_ORDER]
    display(Markdown('## 2. Epoca del best val_total'))
    display(pivot_total_ep)

    pivot_total_last = df_done.pivot(index='arch', columns='opt', values='val_total_last').reindex(ARCH_ORDER)[OPT_ORDER]
    display(Markdown('## 3. val_total a ep30 (se ≈ best, plateau; se ≫ best, sta divergendo)'))
    display(pivot_total_last.round(4))

    # SUB-METRIC (al best_ep_total -- coerenti tra loro)
    display(Markdown('## 4. Sub-metric AL best epoch (val_data, val_phys, val_ou, val_bc)'))
    for sub in ['val_data_at_best', 'val_phys_at_best', 'val_ou_at_best', 'val_bc_at_best']:
        pv = df_done.pivot(index='arch', columns='opt', values=sub).reindex(ARCH_ORDER)[OPT_ORDER]
        display(Markdown(f'### {sub}'))
        display(pv.round(4))

    # SECONDARY: val_data best assoluto (puo' essere a epoca diversa)
    pivot_data = df_done.pivot(index='arch', columns='opt', values='val_data_best').reindex(ARCH_ORDER)[OPT_ORDER]
    display(Markdown('## 5. (Secondary) val_data BEST -- RMSE accel masked, puo\' essere a ep diversa da best_total'))
    display(pivot_data.round(4))

    # Verdetto
    display(Markdown('## 6. Verdetto AdamW vs Prodigy (per arch, su val_total)'))
    best_row = df_done.loc[df_done.val_total_best.idxmin()]
    print(f'BEST ASSOLUTO (val_total): {best_row.tag}')
    print(f'  val_total={best_row.val_total_best:.4f} a ep{best_row.best_ep_total}/{best_row.n_ep}')
    print(f'  Sub: val_data={best_row.val_data_at_best:.4f}  val_phys={best_row.val_phys_at_best:.4f}  '
          f'val_ou={best_row.val_ou_at_best:.4f}  val_bc={best_row.val_bc_at_best:.4f}')
    print()
    for arch in ARCH_ORDER:
        sub = df_done[df_done.arch == arch]
        if len(sub) < 2: continue
        adamw_v = sub[sub.opt == 'adamw'].val_total_best.iloc[0] if (sub.opt == 'adamw').any() else None
        prodigy_v = sub[sub.opt == 'prodigy'].val_total_best.iloc[0] if (sub.opt == 'prodigy').any() else None
        if adamw_v is None or prodigy_v is None: continue
        delta = prodigy_v - adamw_v
        winner = 'AdamW' if delta > 0 else 'Prodigy'
        print(f'  {arch:<20}: AdamW val_total={adamw_v:.4f}  Prodigy val_total={prodigy_v:.4f}  '
              f'delta={delta:+.4f}  -> {winner} vince')

    # Tabella completa runs (per debug / completezza)
    display(Markdown('## 7. Tabella completa runs'))
    display(df_done.set_index('tag')[['n_ep', 'val_total_best', 'best_ep_total', 'val_total_last',
                                        'val_data_at_best', 'val_phys_at_best', 'val_ou_at_best', 'val_bc_at_best',
                                        'val_data_best', 'spike_rate_last', 'prodigy_lr_eff_last']].round(4))

In [ ]:
# ===========================================================
# CELLA 6 -- Trajectory overlay: val_total e val_data per epoca (8 curve x 2 metric)
# ===========================================================
# Top row: val_total (PRIMARY). Bottom row: val_data (secondary, sub-metric).
fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharex=True)
for col, opt_name in enumerate(OPT_ORDER):
    for arch_name in ARCH_ORDER:
        tag = f'T30_{arch_name}_{opt_name}'
        log = f'results/{tag}/training_log.csv'
        if not os.path.isfile(log): continue
        ep = pd.read_csv(log)
        if len(ep) == 0: continue
        axes[0, col].plot(ep.epoch, ep.val_total, 'o-', linewidth=1.7, markersize=4, label=arch_name)
        axes[1, col].plot(ep.epoch, ep.val_data,  's-', linewidth=1.5, markersize=4, label=arch_name)
    axes[0, col].set_title(f'Optimizer: {opt_name.upper()}')
    axes[0, col].set_ylabel('val_total (PINN combined loss)')
    axes[1, col].set_ylabel('val_data (RMSE accel masked)')
    axes[1, col].set_xlabel('epoch')
    for ax in (axes[0, col], axes[1, col]):
        ax.grid(alpha=0.3)
        ax.legend(fontsize=9)
fig.suptitle('val_total (top, PRIMARY) e val_data (bottom, secondary) per epoca -- 4 arch x 2 opt',
             fontsize=12, fontweight='bold')
plt.tight_layout()
os.makedirs('opt_plots', exist_ok=True)
out_traj = 'opt_plots/t30_trajectory_overlay.png'
fig.savefig(out_traj, dpi=120, bbox_inches='tight')
plt.show()
print(f'[saved] {out_traj}')

In [ ]:
# ===========================================================
# CELLA 7 -- Simulator: per ogni best config, run + plot static
# ===========================================================
# Per ogni run completato, carica best_model.pt + esegue CFSimulator su
# scenari val cut-in (cache mixed) per validazione operativa.
from utils.simulator import CFSimulator, compute_operational_metrics, plot_simulation_static

# Cache mixed con cut-in (gia' generata, vedi simulator notebook)
SIM_CACHE = 'data/cache_400_mixed_cut0.3_ou0.0.pt'
if not os.path.isfile(SIM_CACHE):
    print(f'[GEN] {SIM_CACHE} mancante, genero ...')
    from data.generator import generate_dataset
    from config import SEED
    import torch
    train_d = generate_dataset(300, base_seed=SEED, scenario_mix=None,
                                cut_in_ratio=0.30, noise_scale=0.0)
    val_d = generate_dataset(100, base_seed=SEED+1, scenario_mix=None,
                              cut_in_ratio=0.30, noise_scale=0.0)
    torch.save({'train': train_d, 'val': val_d, 'seed': SEED}, SIM_CACHE)
    print(f'[saved] {SIM_CACHE}')

os.makedirs('opt_plots/t30_simviz', exist_ok=True)

# Map ARCH_NAME -> training_method (CFSimulator variant key)
_arch_to_variant = {
    'A1_BASELINE':     'baseline',
    'A8_ATTN':         'attn',
    'A3_STACKED_SKIP': 'stacked_2_skip',
    'EVPROP_ALIF':     'eventprop_alif_full',
}

sim_summaries = []
for p in TRAIN_PLAN:
    tag = p['tag']
    arch = tag.replace('T30_', '').rsplit('_', 1)[0]
    opt  = tag.rsplit('_', 1)[1]
    ckpt = f'checkpoints/{tag}/best_model.pt'
    if not os.path.isfile(ckpt):
        ckpt = f'results/{tag}/best_model.pt'   # fallback
    if not os.path.isfile(ckpt):
        print(f'[SKIP] {tag}: best_model.pt non trovato')
        continue
    try:
        sim = CFSimulator(
            checkpoint_path=ckpt,
            cache_path=SIM_CACHE,
            variant=_arch_to_variant[arch],
            seq_len=200,    # 20s window
        )
    except Exception as e:
        print(f'[ERR] CFSimulator init failed per {tag}: {e}')
        continue
    df_sc = sim.list_scenarios()
    no_cutin_idx = df_sc[~df_sc.is_cut_in].index[0] if (~df_sc.is_cut_in).any() else None
    cutin_idx    = df_sc[df_sc.is_cut_in].index[0]   if df_sc.is_cut_in.any() else None
    for label, idx in [('normal', no_cutin_idx), ('cutin', cutin_idx)]:
        if idx is None: continue
        r = sim.simulate_scenario(int(idx))
        m = compute_operational_metrics(r)
        out_png = f'opt_plots/t30_simviz/{tag}_{label}_idx{idx:03d}.png'
        fig = plot_simulation_static(r, metrics=m)
        fig.savefig(out_png, dpi=100, bbox_inches='tight')
        plt.close(fig)
        # Cross-reference con val_total dal training_log
        val_total_train = None
        sub = df_done[df_done.tag == tag] if 'df_done' in dir() else None
        if sub is not None and len(sub) > 0:
            val_total_train = sub.val_total_best.iloc[0]
        sim_summaries.append({
            'tag': tag, 'label': label, 'idx': int(idx),
            'val_total_train': val_total_train,
            'gap_rmse':   m['gap_rmse_m'],
            'pos_drift':  m['pos_cum_err_m'],
            'accel_rmse': m['accel_rmse_masked'],
            'jerk_max':   m['jerk_max_pred'],
            'spike_avg':  m['spike_rate_avg'],
        })
        print(f'  [{tag} {label}] idx={idx} gap_rmse={m["gap_rmse_m"]:.2f} pos_drift={m["pos_cum_err_m"]:.2f} '
              f'accel_rmse={m["accel_rmse_masked"]:.3f} sr={m["spike_rate_avg"]*100:.1f}% [{out_png}]')
    del sim   # free model

df_sim = pd.DataFrame(sim_summaries)
if len(df_sim) > 0:
    display(Markdown('## Simulator summary (per tag × scenario)'))
    display(df_sim.round(3))
    df_sim.to_csv('opt_plots/t30_simviz/summary.csv', index=False)
    print(f'\n[saved] opt_plots/t30_simviz/summary.csv')